# Day 041 · plotly 交互图 · 中国版
**Plotly** · 阶段 P2 · Python 量化工具栈

> 前面我们用 matplotlib 画的是“死”图——一张图片,看清某一天到底多少钱只能干瞪眼。这一节换一个能动的工具 Plotly:画出来的 K 线能用滚轮缩放、鼠标一悬停就显示当天的开高低收,还能导出成一个网页文件发给朋友,对方双击就能自己玩。我们用真实 A 股宁德时代,从两套写法(px 一行快速、go 一层层搭积木)讲到把主图 K 线和副图成交量拼在一起,最后导出一个可分享的 .html。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-06-10  ·  **建议学习时长:** 18 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "plotly", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 10 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 分清 Plotly 的两套写法:plotly express(px)一行出图适合快速看,graph_objects(go)像搭积木能精细控制每一层
- 会用 go.Candlestick 一行把开高低收画成专业 K 线,再叠加均线
- 会用 make_subplots 把主图 K 线和副图成交量上下排、共用一条时间轴
- 会用 write_html 把交互图导出成一个网页文件,发给任何人都能直接打开缩放、悬停看数值
- 知道交互图和静态图各自的用处:交互图给自己研究和分享,静态图进课件和视频

## 历史背景:老周把 K 线截图发群里,被人怼“这天到底多少钱?”

老周是个10 年股龄的老股民,平时盯盘喜欢截张 K 线图发到股友群里指点江山。有一次他截了一张宁德时代的图,信誓旦旦说“这根大阳线那天起涨”,群友放大图片想看那天具体多少钱,结果像素糊成一团,谁也看不清,反被怼“你倒是说说那天开盘收盘各多少”。老周哑口无言。后来他学了 Plotly,同样的数据画出来是一张能动的图:滚轮一滚就能放大到某一周,鼠标往那根阳线上一停,当天的开盘、收盘、最高、最低四个数字清清楚楚弹出来;他还把图导成一个 .html 网页文件甩进群里,谁点开都能自己缩放查看。从此群里再没人怼他“看不清”——这就是交互式图表的威力:不是一张死图片,而是一个能让别人自己探索的工具。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. px 一行快速 vs go 搭积木

Plotly 有两套写法。plotly express(简称 px)是“偷懒套餐”:px.line(df, x='date', y='Close') 一行就出一张能交互的折线图,适合快速看一眼。graph_objects(简称 go)是“搭积木套餐”:先 go.Figure() 要1 块空白画布,再一层层 add_trace 把 K 线、均线、信号加上去,虽然多写几行,但每一层都能精细控制。规律和前面 matplotlib 的 plt 偷懒 vs ax 精细一模一样:快速看用 px,要做复杂的量价图用 go。


### 2. Candlestick:一行画出专业 K 线

K 线(蜡烛图)一根蜡烛包含一天四个价格:开盘、收盘、最高、最低。自己用线段拼很麻烦,Plotly 直接给了 go.Candlestick,把这四列数据喂进去,一行就出一整张红涨绿跌的专业 K 线,还自带悬停显示四个数值。这就是专门工具的好处:OHLC 这种标准结构,人家替你封装好了。


### 3. make_subplots:主图副图上下排

看盘习惯是上面看价格 K 线、下面看成交量,两块图共用同一条时间轴。make_subplots(rows=2, shared_xaxes=True) 就是开两块上下排列、时间轴对齐的画框,row_heights=[0.7, 0.3] 让主图大、副图小。往哪块画就在 add_trace 里写 row=1 或 row=2。这和 matplotlib 的 subplots 多面板是同一个思路,只是出来的图能交互。


### 4. write_html:导出成可分享的网页

Plotly 图最爽的一点:fig.write_html('k.html') 把整张交互图存成一个独立的网页文件,不依赖任何收费工具、也不用联网。发微信、发邮件给朋友,对方双击用浏览器打开,就能自己缩放、悬停看数值。这是静态图片做不到的——图片发出去就是死的,HTML 发出去是个能玩的工具。


### 5. 交互图 vs 静态图,各有各的用处

交互图(Plotly)适合自己研究和分享:能放大看细节、悬停查数值。但它是网页,塞不进视频和 PDF。所以课件、视频里用的还是静态图(matplotlib 截图)。一个原则:自己看和发给人用交互图,做成品文档用静态图。本节代码两样都生成给你看。


## 实操:Plotly 交互图 — px 一行快速 vs go 搭积木 / Candlestick K 线 / make_subplots 副图 / write_html 导出分享

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_041_plotly.py — Plotly 交互式图表:px 一行快速 vs go 搭积木 / Candlestick K 线 / make_subplots 副图 / 导出 HTML 分享
# 用真实 A 股(宁德时代)做一个能缩放、能悬停看数值的交互式 K 线,并导出成一个 .html 网页发给朋友
# 数据:baostock 日线(免费、稳定、国内零翻墙)
# 依赖:pip install plotly   (静态 png 用 matplotlib 兜底出图,无需 kaleido)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

pd.set_option('display.width', 140)
CODE, NAME = 'sz.300750', '宁德时代'
START, END = '2023-01-01', '2024-12-31'

print('==== 0. 数据拉取(baostock 日线 开高低收 + 成交量)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
rs = bs.query_history_k_data_plus(CODE, 'date,open,high,low,close,volume',
                                  start_date=START, end_date=END, frequency='d', adjustflag='2')
rows = []
while rs.error_code == '0' and rs.next():
    rows.append(rs.get_row_data())
bs.logout()
df = pd.DataFrame(rows, columns=['date', 'Open', 'High', 'Low', 'Close', 'Volume'])
df['date'] = pd.to_datetime(df['date'])
for c in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[c] = df[c].astype(float)
df = df.set_index('date').sort_index()
df['MA20'] = df['Close'].rolling(20).mean()
df['MA60'] = df['Close'].rolling(60).mean()
print(f'{NAME} 日线 {len(df)} 条  {df.index[0].date()} → {df.index[-1].date()}')
print(df[['Open', 'High', 'Low', 'Close', 'Volume']].tail(3).round(2))

# ====================================================================
print('\n==== 1. 两套 API:px 一行快速  vs  go 从空白画布搭积木 ====')
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# px:一句话出一张折线图,适合快速看一眼
fig_px = px.line(df.reset_index(), x='date', y='Close',
                 title=f'{NAME} 收盘价(plotly express 一行快速)')
print('px.line(...) :一行代码就出一张可交互折线图(能缩放、能悬停看数值)')

# go:像搭积木,先要1 块空白画布,再一层层 add_trace 加东西,能精细控制每一层
fig = go.Figure()
fig.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'],
                             low=df['Low'], close=df['Close'], name='K线'))
fig.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='20 日均线', line=dict(width=1.2)))
fig.add_trace(go.Scatter(x=df.index, y=df['MA60'], name='60 日均线', line=dict(width=1.2)))
fig.update_layout(title=f'{NAME} 交互式 K 线 + 双均线', xaxis_rangeslider_visible=False)
print('go.Figure() + add_trace(go.Candlestick(...)):一行 Candlestick 就把开高低收画成一根根 K 线')
print(f'这张 go 图一共叠了 {len(fig.data)} 层:K线 + 两条均线')

# ====================================================================
print('\n==== 2. make_subplots:上面主图 K 线,下面副图成交量,共用一条时间轴 ====')
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     row_heights=[0.72, 0.28], vertical_spacing=0.03,
                     subplot_titles=(f'{NAME} K线 + 均线', '成交量'))
fig2.add_trace(go.Candlestick(x=df.index, open=df['Open'], high=df['High'],
                              low=df['Low'], close=df['Close'], name='K线'), row=1, col=1)
fig2.add_trace(go.Scatter(x=df.index, y=df['MA20'], name='20 日均线', line=dict(width=1.1)), row=1, col=1)
up = df['Close'] >= df['Open']
fig2.add_trace(go.Bar(x=df.index[up], y=df['Volume'][up], name='放量上涨',
                      marker_color='#bf616a'), row=2, col=1)
fig2.add_trace(go.Bar(x=df.index[~up], y=df['Volume'][~up], name='放量下跌',
                      marker_color='#3b8a5a'), row=2, col=1)
fig2.update_layout(xaxis_rangeslider_visible=False, showlegend=False,
                   title=f'{NAME} 量价交互图(主图 K 线 + 副图成交量)')
print('make_subplots(rows=2, shared_xaxes=True):主图、副图上下排,时间轴对齐')
print('副图用涨绿跌红两种颜色的成交量柱,放量上涨/放量下跌一眼可分')

# ====================================================================
print('\n==== 3. 导出 HTML(真正的成品,鼠标悬停看数值、滚轮缩放、发给朋友直接打开)====')
html_path = f'D041_{NAME}_交互K线.html'
fig2.write_html(html_path)
print(f'交互式网页已导出:{html_path}(双击用浏览器打开就能缩放、悬停看每天的开高低收)')
print('write_html 只需要 plotly 本身,不依赖任何收费工具,发微信/邮件给朋友都能直接看')

# ====================================================================
# 课件/视频需要静态图。plotly 的交互图导出 png 需要额外的 kaleido;
# 这里用 matplotlib 复刻同样的 K 线,稳定出图(run_lab 自动捕获 chart_NN.png)。
print('\n==== 4. 静态截图(matplotlib 复刻,供课件/视频展示交互图长什么样)====')
plt.rcParams['axes.unicode_minus'] = False


def mpl_candle(ax, d):
    """用 matplotlib 画一版 K 线:细线是上下影线,粗线是实体,红涨绿跌。"""
    rise = d['Close'] >= d['Open']
    ax.vlines(d.index, d['Low'], d['High'], color='#999', lw=0.5)
    ax.vlines(d.index[rise], d['Open'][rise], d['Close'][rise], color='#bf616a', lw=2.2)
    ax.vlines(d.index[~rise], d['Open'][~rise], d['Close'][~rise], color='#3b8a5a', lw=2.2)


# 图1:K 线 + 双均线(对应上面 go.Figure 的成品)
fig_a, ax = plt.subplots(figsize=(13, 5.5))
mpl_candle(ax, df)
ax.plot(df.index, df['MA20'], color='#d08770', lw=1.3, label='20 日均线')
ax.plot(df.index, df['MA60'], color='#5e81ac', lw=1.3, label='60 日均线')
ax.set_title(f'{NAME} 交互式 K 线 + 双均线(静态截图)'); ax.set_ylabel('价格(元)'); ax.legend()
plt.tight_layout(); plt.savefig('chart_01.png', dpi=110); plt.close()
print('图1(K 线 + 双均线)已保存')

# 图2:主图 K 线 + 副图成交量(对应 make_subplots 的成品)
fig_b, (axp, axv) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                                 gridspec_kw={'height_ratios': [3, 1]})
mpl_candle(axp, df)
axp.plot(df.index, df['MA20'], color='#d08770', lw=1.1, label='20 日均线')
axp.set_title(f'{NAME} 量价图:主图 K 线 / 副图成交量'); axp.set_ylabel('价格(元)'); axp.legend()
axv.bar(df.index[up], df['Volume'][up] / 1e6, color='#bf616a', width=1.0)
axv.bar(df.index[~up], df['Volume'][~up] / 1e6, color='#3b8a5a', width=1.0)
axv.set_ylabel('成交量(百 万股)')
plt.tight_layout(); plt.savefig('chart_02.png', dpi=110); plt.close()
print('图2(主图 K 线 + 副图成交量)已保存')

# 图3:px 一行折线(对应 px.line 的快速看一眼)
fig_c, axc = plt.subplots(figsize=(12, 4.5))
axc.plot(df.index, df['Close'], color='#5e81ac', lw=1.2)
axc.set_title(f'{NAME} 收盘价(px.line 一行快速看一眼)'); axc.set_ylabel('价格(元)')
plt.tight_layout(); plt.savefig('chart_03.png', dpi=110); plt.close()
print('图3(px 一行折线)已保存')

# 小结数字(供文案核对)
period_ret = df['Close'].iloc[-1] / df['Close'].iloc[0] - 1
print(f'\n[小结] {NAME} 区间 {df.index[0].date()}→{df.index[-1].date()} 涨跌幅 {period_ret*100:.1f}%')
print(f'最高 {df["High"].max():.2f} 元 / 最低 {df["Low"].min():.2f} 元 / 期末 {df["Close"].iloc[-1]:.2f} 元')
print('\n[done] 交互式 HTML 1 个 + 静态图 3 张已生成')

==== 0. 数据拉取(baostock 日线 开高低收 + 成交量)====
login success!
logout success!
宁德时代 日线 484 条  2023-01-03 → 2024-12-31
              Open    High     Low   Close      Volume
date                                                  
2024-12-27  249.41  252.14  244.08  250.78  24351918.0
2024-12-30  249.71  256.04  249.35  255.16  22631352.0
2024-12-31  253.74  258.92  253.74  254.61  24933952.0

==== 1. 两套 API:px 一行快速  vs  go 从空白画布搭积木 ====
px.line(...) :一行代码就出一张可交互折线图(能缩放、能悬停看数值)
go.Figure() + add_trace(go.Candlestick(...)):一行 Candlestick 就把开高低收画成一根根 K 线
这张 go 图一共叠了 3 层:K线 + 两条均线

==== 2. make_subplots:上面主图 K 线,下面副图成交量,共用一条时间轴 ====
make_subplots(rows=2, shared_xaxes=True):主图、副图上下排,时间轴对齐
副图用涨绿跌红两种颜色的成交量柱,放量上涨/放量下跌一眼可分

==== 3. 导出 HTML(真正的成品,鼠标悬停看数值、滚轮缩放、发给朋友直接打开)====
交互式网页已导出:D041_宁德时代_交互K线.html(双击用浏览器打开就能缩放、悬停看每天的开高低收)
write_html 只需要 plotly 本身,不依赖任何收费工具,发微信/邮件给朋友都能直接看

==== 4. 静态截图(matplotlib 复刻,供课件/视频展示交互图长什么样)====
图1(K 线 + 双均线)已保存
图2(主图 K 线 + 副图成交量)已保存
图3(px 一行折线)已保存

[小结] 宁德时代 区间 2023-01-03→202

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 交互式 K 线 | sz.300750 | 宁德时代日线做一张能缩放、能悬停看开高低收的交互式 K 线,导出 HTML 发到股友群 |
| A 股复盘 |  | 复盘某天大阳线时,鼠标悬停直接读出当天开盘收盘最高最低,不用再去翻行情软件 |
| 量价分析 |  | 主图 K 线 + 副图成交量上下排,放量上涨和放量下跌用红绿成交量柱一眼区分 |
| 策略分享 |  | 把回测净值曲线导成 HTML 发给同伴,对方自己缩放查看每个回撤段,不用来回截图 |


## 常见坑

### ⚠ 01. px 和 go 混着用搞晕自己

px 返回的图和 go 搭的图都是 Figure,但写法风格不同。新手常常 px 出一半又想用 go 的细节控制,代码乱成一团。建议:简单快速看用 px,正经做量价图就从头用 go,别中途换风格。

### ⚠ 02. K 线带了一条碍事的迷你滑块

go.Candlestick 默认在图下方带一条 rangeslider 迷你缩略图,很多时候碍事还压缩主图空间。一句 fig.update_layout(xaxis_rangeslider_visible=False) 关掉它,主图立刻清爽。

### ⚠ 03. 想导出 png 却报缺 kaleido

Plotly 导出静态图片(write_image)要额外装一个叫 kaleido 的引擎,没装就报错。但导出 HTML(write_html)只要 plotly 本身,不需要 kaleido。所以分享用 HTML 最省事;真要 png,要么装 kaleido,要么像本节一样用 matplotlib 复刻一张。

### ⚠ 04. 把交互图硬塞进 PDF / 视频

Plotly 是网页,塞不进 PDF 和视频里。想在文档、视频里放图,必须用静态图片(matplotlib 或 plotly+kaleido 导出的 png)。别指望把 .html 贴进 Word 还能交互。

### ⚠ 05. 数据列名喂错导致 K 线画不出

go.Candlestick 要分别喂 open / high / low / close 四列,列名或顺序搞错,K 线就画歪甚至报错。拉数据后先确认四列是 float、且对应正确,再喂给 Candlestick。

## 实战 SOP · Plotly 交互作图实战 6 条 SOP

1. 快速看一眼用 px,正经量价图用 go — 别中途混用两套风格。
2. K 线一行 go.Candlestick — 开高低收四列喂进去,确认都是 float。
3. 默认关掉 rangeslider — update_layout(xaxis_rangeslider_visible=False) 让主图清爽。
4. 主副图用 make_subplots — rows=2 + shared_xaxes=True,时间轴对齐。
5. 分享首选 write_html — 只需 plotly,导出网页发给任何人都能打开。
6. 课件视频用静态图 — 交互图进不了 PDF/视频,用 matplotlib 或 kaleido 出 png。

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. Plotly 画的是“能动”的图:滚轮缩放、鼠标悬停看数值,和 matplotlib 的死图片完全不同。
3. 两套写法:px 一行快速看一眼,go 搭积木精细控制——和 matplotlib 的 plt vs ax 一个道理。
4. go.Candlestick 一行把开高低收画成专业 K 线,再 add_trace 叠均线。
5. make_subplots(rows=2, shared_xaxes=True) 主图 K 线 + 副图成交量,时间轴对齐。
6. write_html 把交互图导出成网页文件,只需 plotly,发给谁都能打开自己玩。
7. 导出 png 需要额外的 kaleido;课件、视频用静态图,交互图自己研究和分享用。

## 自测题

**Q1.** Plotly 的 px 和 go 有什么区别,各自什么时候用?(提示:px 是 plotly express,一行出图适合快速看一眼;go 是 graph_objects,先开空白 Figure 再一层层 add_trace,能精细控制每一层,做复杂量价图用它。和 matplotlib 的 plt 偷懒 vs ax 面向对象是同一个道理。)

**Q2.** 为什么 Plotly 画 K 线比 matplotlib 省事?(提示:K 线要表达一天开高低收四个价格,matplotlib 得自己拼线段画实体和影线;Plotly 直接提供 go.Candlestick,四列数据喂进去一行就出红涨绿跌的专业 K 线,还自带悬停显示数值。)

**Q3.** 怎么把主图 K 线和副图成交量上下排、时间轴对齐?(提示:用 make_subplots(rows=2, cols=1, shared_xaxes=True) 开两块上下画框,row_heights 控制主大副小;add_trace 时用 row=1 / row=2 指定画到哪块。)

**Q4.** 想把图发给朋友自己缩放查看,该用什么?需要额外装东西吗?(提示:用 fig.write_html('x.html') 导出成网页文件,只依赖 plotly 本身,不需要 kaleido,对方双击浏览器打开就能交互。导出静态 png 才需要装 kaleido。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 042 · mplfinance 画 K 线** (mplfinance)

下一节我们换一个专门为 K 线而生的工具 mplfinance:一行 type='candle' 就出专业蜡烛图,均线、成交量副图全自动,还能一键换主题,看看比 matplotlib 手画省多少事。

## 推荐阅读

- Plotly 官方文档《Candlestick Charts in Python》(2024)— go.Candlestick 的标准用法与样式参数
- Plotly 官方文档《Subplots in Python》(2024)— make_subplots 多面板、共享坐标轴权威参考
- Plotly 官方文档《Interactive HTML Export》(2024)— write_html 导出与分享交互图的说明
- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— pandas 数据整理为绘图做准备
- Carson Sievert《Interactive Web-Based Data Visualization with R, plotly, and shiny》(2020, CRC Press)— 交互可视化的设计思想,理念跨语言通用